In [1]:
# Import Libraries

import gc
import sys
import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
import matplotlib.pyplot as plt

import golois

print ("Python version", sys.version_info)
print ("Tensorflow version", tf.__version__)


Python version sys.version_info(major=3, minor=9, micro=21, releaselevel='final', serial=0)
Tensorflow version 2.15.0


In [2]:
# Configuration

planes = 31
moves = 361
N = 25000


input_data = np.random.randint(2, size=(N, 19, 19, planes))
input_data = input_data.astype ('float32')

policy = np.random.randint(moves, size=(N,))
policy = keras.utils.to_categorical (policy)

value = np.random.randint(2, size=(N,))
value = value.astype ('float32')

end = np.random.randint(2, size=(N, 19, 19, 2))
end = end.astype ('float32')

groups = np.zeros((N, 19, 19, 1))
groups = groups.astype ('float32')

In [3]:
# Get Validation Data

print ("getValidation", flush = True)
golois.getValidation (input_data, policy, value, end)

getValidation


r.shape = (25000, 19, 19, 31)
nbExamples = 25000
nbPositionsSGF = 102208897
nbPositionsSGF = 102208897
loading validation.data


In [4]:
#from tensorflow.keras.utils import plot_model
#from IPython.display import Image

from tensorflow import keras
from keras import regularizers
from keras.models import Model
from keras.layers import (
    Input, Dense, Conv2D, GlobalAveragePooling2D, Dropout, Flatten,
    Activation, BatchNormalization, Add, Reshape, DepthwiseConv2D, LeakyReLU,
    Multiply
)
from keras.utils import plot_model

def _se_block(input_tensor, filters, ratio=16, activation='relu'):
    se = GlobalAveragePooling2D()(input_tensor)
    se = Reshape((1, 1, filters))(se)
    se = Dense(filters // ratio, activation=activation, use_bias=False)(se)
    se = Dense(filters, activation='sigmoid', use_bias=False)(se)
    return Multiply()([input_tensor, se])

def _conv_block(inputs, filters, kernel, activation='relu'):
    x = Conv2D(filters, kernel, padding='same', kernel_regularizer=regularizers.l2(0.0001), use_bias=False)(inputs)
    x = BatchNormalization(axis=1)(x)
    x = Activation(activation)(x)
    return x

def _bottleneck_block(inputs, filters, kernel, factor, se, activation='relu'):
    expanded_filters = filters * factor

    x = _conv_block(inputs, filters=expanded_filters, kernel=(1, 1), activation=activation)
    x = DepthwiseConv2D(kernel, padding='same', kernel_regularizer=regularizers.l2(0.0001), use_bias=False)(x)
    x = BatchNormalization(axis=1)(x)
    x = _conv_block(x, filters=filters, kernel=(1, 1), activation=activation)

    if se:
        x = _se_block(x, filters, 16, activation)

    x = Add()([x, inputs])
    x = Activation(activation)(x)
    return x

def GoMobileNetv2(input_shape, filters, factor, block_num, se, activation='relu', drop_out_rate=0.3):
    inputs = Input(shape=input_shape)
    x = _conv_block(inputs, filters, (1, 1), activation=activation)

    for i in range(block_num):
        x = _bottleneck_block(x, filters, (3, 3), factor, se, activation=activation)

    policy_head = _conv_block(x, filters=1, kernel=(1, 1), activation=activation)
    policy_head = Flatten()(policy_head)
    policy_head = Activation('softmax', name='policy')(policy_head)

    value_head = GlobalAveragePooling2D()(x)
    value_head = Dense(50, kernel_regularizer=regularizers.l2(0.0001))(value_head)
    value_head = Activation(activation)(value_head)
    value_head = Dropout(drop_out_rate)(value_head)
    value_head = Dense(1, activation='sigmoid', name='value',
                       kernel_regularizer=regularizers.l2(0.0001))(value_head)

    model = keras.Model(inputs=inputs, outputs=[policy_head, value_head])
    return model

def GoMobileNetv3(input_shape, factor, se, activation='relu'):

    inputs = Input(shape=input_shape)
    x = _conv_block(inputs, 32, (1, 1), activation=activation)

    x = _conv_block(x, filters=16, kernel=(1, 1), activation=activation)
    x = _bottleneck_block(x, 16, (3,3), factor, se, activation=activation)

    x = _conv_block(x, filters=32, kernel=(1, 1), activation=activation)
    x = _bottleneck_block(x, 32, (3,3), factor, se, activation=activation)

    x = _conv_block(x, filters=96, kernel=(1, 1), activation=activation)
    x = _bottleneck_block(x, 96, (3,3), factor, se, activation=activation)

    # Policy head
    policy_head = _conv_block(x, filters=1, kernel=(1, 1), activation=activation)
    policy_head = Flatten()(policy_head)
    policy_head = Activation('softmax', name='policy')(policy_head)

    # Value head
    value_head = GlobalAveragePooling2D()(x)
    value_head = Dense(50, kernel_regularizer=regularizers.l2(0.0001))(value_head)
    value_head = Activation(activation)(value_head)
    value_head = Dropout(0.3)(value_head)
    value_head = Dense(1, activation='sigmoid', name='value',
                              kernel_regularizer=regularizers.l2(0.0001))(value_head)

    model = keras.Model(inputs=inputs, outputs=[policy_head, value_head])
    return model

def GoMobileNetv4(input_shape, factor, se, activation='relu'):

    inputs = Input(shape=input_shape)
    x = _conv_block(inputs, 32, (1, 1), activation=activation)

    x = _conv_block(x, filters=16, kernel=(1, 1), activation=activation)
    x = _bottleneck_block(x, 16, (3,3), factor, se, activation=activation)

    x = _conv_block(x, filters=32, kernel=(1, 1), activation=activation)
    x = _bottleneck_block(x, 32, (5,5), factor, se, activation=activation)

    x = _conv_block(x, filters=64, kernel=(1, 1), activation=activation)
    x = _bottleneck_block(x, 64, (7,7), factor, se, activation=activation)

    # Policy head
    policy_head = _conv_block(x, filters=1, kernel=(1, 1))
    policy_head = Flatten()(policy_head)
    policy_head = Activation('softmax', name='policy')(policy_head)

    # Value head
    value_head = GlobalAveragePooling2D()(x)
    value_head = Dense(50, kernel_regularizer=regularizers.l2(0.0001))(value_head)
    value_head = LeakyReLU()(value_head)
    value_head = Activation(activation)(value_head)
    value_head = Dropout(0.3)(value_head)
    value_head = Dense(1, activation='sigmoid', name='value',
                              kernel_regularizer=regularizers.l2(0.0001))(value_head)

    model = keras.Model(inputs=inputs, outputs=[policy_head, value_head])
    return model

#model = GoMobileNetv3((19,19,31), 4, True)
#odel.summary()


# Sauvegarde du schéma au format PNG
#plot_model(model, to_file='model.png', show_shapes=True, show_layer_names=True)
# Affichage dans le notebook
#Image(filename='model.png')


In [5]:
import time
import pandas as pd
from tensorflow.keras import optimizers, callbacks, backend as K

# Epochs number
epochs = 500

# Cosine Annealing function
def get_cosine_annealing_lr(epoch, base_lr=0.0005, min_lr=0.0000005, total_epochs=epochs):
    cos_inner = np.pi * (epoch % total_epochs) / total_epochs
    lr = min_lr + 0.5 * (base_lr - min_lr) * (1 + np.cos(cos_inner))
    print(f"[Cosine Annealing] Epoch {epoch+1}/{total_epochs} - lr: {lr:.8f}")
    return lr

# Scheduler de learning rate (comme dans le papier)
def get_learning_rate(epoch):
    if epoch < (epochs/5)*2.5:
        return 0.0005
    elif epoch < (epochs/5)*3:
        return 0.00005
    elif epoch < (epochs/5)*4:
        return 0.000005
    else:
        return 0.0000005

def train_model(model, batch=32, scheduler_type="cosine", policy_weight = 1.0, 
                value_weight = 1.0, epochs=epochs, validation_split=0.1):
  # Démarrer le chrono
  start_time = time.time()

  # Optimiseur : SGD avec momentum
  # Choix du scheduler
  if scheduler_type == "cosine":
        lr = get_cosine_annealing_lr(0)
  elif scheduler_type == "step":
        lr = get_learning_rate(0)
  optimizer = optimizers.legacy.SGD(learning_rate=lr, momentum=0.9, nesterov=True)

  # Pour stocker les métriques de chaque epoch
  all_history = []

  # Compilation du modèle
  model.compile(optimizer=optimizer,
                loss={'policy': 'categorical_crossentropy', 'value': 'binary_crossentropy'},
                loss_weights={'policy': policy_weight, 'value': value_weight},
                metrics={'policy': 'categorical_accuracy', 'value': 'mse'})

  # Entraînement avec scheduler
  for i in range(1, epochs + 1):

      # Choix du scheduler
      if scheduler_type == "cosine":
          lr = get_cosine_annealing_lr(i)
      elif scheduler_type == "step":
          lr = get_learning_rate(i)
      else:
          raise ValueError(f"Scheduler type inconnu : {scheduler_type}")

      keras.backend.set_value(model.optimizer.learning_rate, lr)
      print('epoch ' + str(i)+ ', lr '+str(lr))

      # ➡️ Générer de nouveaux batchs avec symétrie
      golois.getBatch(input_data, policy, value, end, groups, i * N)

      history = model.fit(input_data,
                          {'policy': policy, 'value': value},
                          epochs=1,
                          batch_size=batch,
                          verbose=1,
                          validation_split=validation_split)

      # Stocker l’historique
      metrics = {key: val[0] for key, val in history.history.items()}
      metrics['epoch'] = i
      all_history.append(metrics)

      if i % 5 == 0:
          gc.collect()

      if i % epochs == 0:
        golois.getValidation(input_data, policy, value, end)
        val = model.evaluate(input_data,
                              [policy, value], verbose=0, batch_size=batch)
        print(f"Validation - policy: {val[1]:.4f}, value: {val[2]:.4f}")
        model.save ('mchettih.h5')

  total_time = time.time() - start_time
  return val, pd.DataFrame(all_history), total_time

In [6]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def plot_result(history_dfs, labels, epochs=None):
    assert len(history_dfs) == len(labels)

    # Titre
    info = []
    title = f"Epochs: {epochs}"

    # Grille personnalisée : 2 lignes (3 en haut, 2 en bas)
    fig = plt.figure(figsize=(18, 8))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    gs = gridspec.GridSpec(2, 3)

    # --- Ligne 1 : 3 plots ---
    ax1 = fig.add_subplot(gs[0, 0])
    for df, label in zip(history_dfs, labels):
        ax1.plot(df['epoch'], df['loss'], label=f'{label} Total Loss')
    ax1.set_title('Total Loss par Epoch')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Total Loss')
    ax1.legend()

    ax2 = fig.add_subplot(gs[0, 1])
    for df, label in zip(history_dfs, labels):
        if 'policy_loss' in df.columns:
            ax2.plot(df['epoch'], df['policy_loss'], label=f'{label} Policy Loss')
    ax2.set_title('Policy Loss par Epoch')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Policy Loss')
    ax2.legend()

    ax3 = fig.add_subplot(gs[0, 2])
    for df, label in zip(history_dfs, labels):
        if 'value_loss' in df.columns:
            ax3.plot(df['epoch'], df['value_loss'], label=f'{label} Value Loss')
    ax3.set_title('Value Loss par Epoch')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Value Loss')
    ax3.legend()

    # --- Ligne 2 : 2 plots ---
    ax4 = fig.add_subplot(gs[1, 0])
    for df, label in zip(history_dfs, labels):
        if 'policy_categorical_accuracy' in df.columns:
            ax4.plot(df['epoch'], df['policy_categorical_accuracy'], label=label)
    ax4.set_title('Policy Accuracy par Epoch')
    ax4.set_xlabel('Epoch')
    ax4.set_ylabel('Categorical Accuracy')
    ax4.legend()

    ax5 = fig.add_subplot(gs[1, 1])
    for df, label in zip(history_dfs, labels):
        if 'value_mse' in df.columns:
            ax5.plot(df['epoch'], df['value_mse'], label=label)
    ax5.set_title('Value MSE par Epoch')
    ax5.set_xlabel('Epoch')
    ax5.set_ylabel('MSE')
    ax5.legend()

    # Libérer la dernière case vide de la 2e ligne
    fig.delaxes(fig.add_subplot(gs[1, 2]))  # case vide propre

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()

def print_validation_results(model_results, epoch=epochs):
    """
    Affiche les résultats de validation pour une liste de modèles.

    Paramètres :
    - model_results : liste de tuples (model, val, label)
    - epoch : (optionnel) numéro d'epoch à afficher dans le titre
    """
    for model, val, label, time in model_results:
        metrics = dict(zip(model.metrics_names, val))
        title = f"📊 Validation Results for {label}"
        if epoch is not None:
            title += f" — Epoch {epoch}"
        print(f"\n{title}:")
        for name, value in metrics.items():
            print(f"  - {name:<30}: {value:.4f}")
        print(f"  - Time: {time:.4f}")

In [7]:
# Modèle 1 : GoMobileNetv2 (64,4,3) 
# - Nesterov  True 
# - SE True
# - Cosine Annealing
# - Batch size 32
# - Policy weight 1.0
# - Value weight 4.0

model_1 = GoMobileNetv2((19,19,31), 64, 4, 3,  True, activation='swish')
val_1, all_history_1, total_time_1 = train_model(model_1, 
                                                 batch=32, 
                                                 scheduler_type="step", 
                                                 policy_weight = 1.0, 
                                                 value_weight = 1.0,
                                                 epochs=500)

#model_2 = GoMobileNetv2((19,19,31), 64, 4, 3,  True, activation='swish')
#val_2, all_history_2, total_time_2 = train_model(model_2, 
#                                                 batch=32, 
#                                                 scheduler_type="cosine", 
#                                                 policy_weight = 4.0, 
#                                                 value_weight = 1.0,
#                                                 epochs=100)

#model_3 = GoMobileNetv2((19,19,31), 64, 4, 3,  True, activation='swish')
#val_3, all_history_3, total_time_3 = train_model(model_3, 
#                                                 batch=32, 
#                                                 scheduler_type="cosine", 
#                                                 policy_weight = 1.0, 
#                                                 value_weight = 4.0,
#                                                 epochs=100)

# Affichage des résultats
results = [
    (model_1, val_1, "Model 1", total_time_1)
]
print_validation_results(results)

# Affichage des courbes comparatives
plot_result(
    history_dfs=[all_history_1],
    labels=["Model 1"],
    epochs=epochs
)

2025-04-26 00:51:33.403492: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2025-04-26 00:51:33.403523: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 48.00 GB
2025-04-26 00:51:33.403529: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 18.00 GB
2025-04-26 00:51:33.403563: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-04-26 00:51:33.403583: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


epoch 1, lr 0.0005


r.shape = (25000, 19, 19, 31)
nbExamples = 25000
2025-04-26 00:51:35.679272: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


704/704 [==============================] - 31s 42ms/step - loss: 5.5919 - policy_loss: 4.8246 - value_loss: 0.6957 - policy_categorical_accuracy: 0.1100 - value_mse: 0.1227 - val_loss: 4.9534 - val_policy_loss: 4.1907 - val_value_loss: 0.6912 - val_policy_categorical_accuracy: 0.2084 - val_value_mse: 0.1210
epoch 2, lr 0.0005


r.shape = (25000, 19, 19, 31)
nbExamples = 25000


...

r.shape = (25000, 19, 19, 31)
nbExamples = 25000


704/704 [==============================] - 33s 47ms/step - loss: 3.2840 - policy_loss: 2.5354 - value_loss: 0.6855 - policy_categorical_accuracy: 0.3827 - value_mse: 0.1175 - val_loss: 3.2384 - val_policy_loss: 2.4872 - val_value_loss: 0.6880 - val_policy_categorical_accuracy: 0.3972 - val_value_mse: 0.1212
epoch 215, lr 0.0005


r.shape = (25000, 19, 19, 31)
nbExamples = 25000


253/704 [=========>....................] - ETA: 19s - loss: 3.2694 - policy_loss: 2.5197 - value_loss: 0.6865 - policy_categorical_accuracy: 0.3855 - value_mse: 0.1179

KeyboardInterrupt: 

In [ ]:
#Test increasing nb filter in layers

model_2 = GoMobileNetv2((19,19,31), 64, 4, 3,  True)
val_2, all_history_2, total_time_2 = train_model(model_2)

model_3 = GoMobileNetv3((19,19,31), 4, True)
val_3, all_history_3, total_time_3 = train_model(model_3)

# Affichage des résultats
results = [
    (model_2, val_2, "Mobile Fixe Filters Nb", total_time_2),
    (model_3, val_3, "Mobile Mix Filters Nb", total_time_3)
]
print_validation_results(results)

# Affichage des courbes comparatives
plot_result(
    history_dfs=[all_history_2, all_history_3],
    labels=["Mobile Fixe Filters Nb", "Mobile MIX Filters Nb"],
    epochs=epochs
)

In [ ]:
#Test increasing filter size (3x3) (5x5) (7x7) in layers

#model_2 = GoMobileNetv2((19,19,31), 64, 4, 3,  True)
#al_2, all_history_2, total_time_2 = train_model(model_2)

model_4 = GoMobileNetv4((19,19,31), 4, True)
val_4, all_history_4, total_time_4 = train_model(model_4)

# Affichage des résultats
results = [
    (model_2, val_2, "Mobile Fixe Filter Size", total_time_2),
    (model_3, val_3, "Mobile Increase Filter Size", total_time_4)
]
print_validation_results(results)

# Affichage des courbes comparatives
plot_result(
    history_dfs=[all_history_2, all_history_4],
    labels=["Mobile Fixe Filter Size", "Mobile Increase Filter Size"],
    epochs=epochs
)


In [ ]:
# Test Of Dropout rates
drop_out_rates = [0.2, 0.3, 0.5]

results = []
histories = []

for drop_out_rate in drop_out_rates:
    model = GoMobileNetv2((19, 19, 31), filters=64, factor=4, block_num=3, se=True, activation='swish', drop_out_rate=drop_out_rate)
    val_result, history_df, training_time = train_model(model, epochs=50)
    results.append((model, val_result, f"MobileNetv2 DropOut {drop_out_rate}", training_time))
    histories.append(history_df)

# Affichage des résultats comparatifs
print_validation_results(results)

# Affichage des courbes d'apprentissage
labels = [label for (_, _, label, _) in results]
plot_result(
    history_dfs=histories,
    labels=labels,
    epochs=50
)

In [ ]:
# Test Of Validation splits
validation_splits = [0.1, 0.2]

results = []
histories = []

for validation_split in validation_splits:
    model = GoMobileNetv2((19, 19, 31), filters=64, factor=4, block_num=3, se=True, activation='swish')
    val_result, history_df, training_time = train_model(model, epochs=50, validation_split=validation_split)
    results.append((model, val_result, f"MobileNetv2 Validation Split {validation_split}", training_time))
    histories.append(history_df)

# Affichage des résultats comparatifs
print_validation_results(results)

# Affichage des courbes d'apprentissage
labels = [label for (_, _, label, _) in results]
plot_result(
    history_dfs=histories,
    labels=labels,
    epochs=50
)